<a href="https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/ColabFold2_preview.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ColabFold2 preview

Protein, RNA, DNA and small-molecule structures with
[AlphaFold 3](https://www.nature.com/articles/s41586-024-07487-w) and thirteen other
sets of weights, through one implementation. MSAs come from the
[ColabFold](https://github.com/sokrypton/ColabFold) MMseqs2 server, so no databases
are needed.

**Please cite** AlphaFold 3 ([Abramson 2024](https://doi.org/10.1038/s41586-024-07487-w)),
ColabFold ([Mirdita 2022](https://doi.org/10.1038/s41592-022-01488-1)) and whichever
model you ran.


In [ ]:
#@title Install dependencies (~35 s)
import os, time, glob, shutil, sys, subprocess
_T0 = time.time()

model = "openbind0" #@param ["openbind0", "openfold3", "boltz2", "protenix2", "rosettafold3", "chai1", "intellifold2", "opendde", "esmfold2", "esmfold2_lm600m", "esmfold2_lm300m", "alphafold3", "af2_ptm", "af2_multimer"]

persist_cache_to_drive = False #@param {type:"boolean"}
#@markdown - `persist_cache_to_drive` keeps the compiled model in Drive, so the
#@markdown   next session skips the compile: 72 s to 26 s on a T4. Never changes
#@markdown   a result.

# Set any form field from the environment, for runs outside Colab:
#   AF3_NB_OVERRIDES='{"model": "boltz2"}'
import json as _json
for _k, _v in _json.loads(os.environ.get('AF3_NB_OVERRIDES', '{}')).items():
  if _k in globals():
    globals()[_k] = _v
    print(f'override: {_k} = {_v!r}')

VERSION = '3.1.11'          # the wheel, from PyPI
# Where the PYTHON comes from. A tag (`v3.1.11`) is the shipping notebook; the
# `colab` branch carries the experimental live-animation and steering code,
# which no release has. See the overlay below.
SOURCE = 'colab'
NATIVE_DIR = 'af3_native_weights'
AF3_WEIGHTS_URL = 'https://storage.googleapis.com/alphafold3/af3.bin.zst'
AF2_DIR = 'af2_params'
IS_AF3 = (model == 'alphafold3')
IS_AF2 = model.startswith('af2_')
# int8 weights, expanded on load; AF2 and AF3 ship their own float32 files.
PRECISION = 'fp32' if (IS_AF3 or IS_AF2) else 'int8'


def _sh(cmd, what):
  """Run a shell command, raising if it fails."""
  if os.system(cmd) != 0:
    raise RuntimeError(f'{what} failed. The output is above.')


if not os.path.isfile('ALPHAFOLD3_READY'):
  print('Installing packages...')
  # Installed with --no-deps, so the package's own imports are listed here.
  # Letting pip resolve them would re-download jax and the CUDA stack.
  # tokamax 0.0.11 WORKS ON jax 0.11.1 AND BREAKS ON 0.11.2, where
  # `jax.experimental.hijax.HiPrimitive` is gone -- and with tokamax
  # unimportable, nothing in the stack loads. A GPU image already has 0.11.1,
  # satisfies tokamax's `jax>=0.9.1`, and pip leaves it alone; a TPU image has
  # 0.7.2, FAILS that requirement, and pip upgrades it to the latest.
  # So LOOK BEFORE INSTALLING. Re-pinning a jax that is already right costs
  # 38 s and a libtpu download for nothing, and on a GPU image touching jax at
  # all would drag in the CUDA stack -- which is the very thing --no-deps is
  # here to avoid.
  import glob as _glob
  import importlib.metadata as _md
  try:
    _jax_now = _md.version('jax')
  except Exception:
    _jax_now = None
  _is_tpu = bool(_glob.glob('/dev/accel*') or os.environ.get('TPU_ACCELERATOR_TYPE'))
  if _jax_now == '0.11.1':
    pass                     # every GPU image today: nothing to do
  elif _is_tpu:
    print(f'jax {_jax_now} on a TPU runtime; pinning to 0.11.1 for tokamax')
    _sh('pip install -q "jax[tpu]==0.11.1"', 'pinning jax for the TPU runtime')
  else:
    # Deliberately NOT fixed here: jax[cuda12] would re-download the CUDA
    # stack. Say it plainly instead -- if the Colab GPU image ever moves to
    # 0.11.2, this line is what explains the import errors that follow.
    print(f'NOTE: jax {_jax_now} is not the 0.11.1 tokamax 0.0.11 expects; '
          'if imports fail with hijax.HiPrimitive, pin jax to 0.11.1.')
  # A PRE-AMPERE CARD HAS NO OTHER FUSED ATTENTION. cuDNN's SDPA wants SM80,
  # tokamax has no kernel for it, and XLA gates Pallas/Triton at sm_80 -- so a
  # T4, the commonest Colab GPU, runs the materialising XLA path for the 77%
  # of a pairformer pass that is triangle attention. Milot Mirdita's
  # colabfold-legacy-kernels is the exception: 3.0-3.35x that path on a T4,
  # measured at this model's own shape. Linux x86_64 wheels, and only where
  # the card can use them.
  import platform as _pyplat
  _cc0 = None
  if _pyplat.system() == 'Linux' and _pyplat.machine() == 'x86_64':
    try:
      _out0 = subprocess.run(['nvidia-smi', '--query-gpu=compute_cap',
                              '--format=csv,noheader'],
                             capture_output=True, text=True, timeout=15).stdout
      _cc0 = min(float(x) for x in _out0.split() if x.strip())
    except Exception:
      _cc0 = None
  if _cc0 is not None and _cc0 < 8.0:
    # BOTH PACKAGES: the prebuilt .so files are in the first and the wrapper
    # that dlopens them, registers the FFI symbols and holds the custom_vjp
    # is `colabfold_kernels.volta` in the second. Installing only the first
    # is not an error -- the model falls back to XLA and folds 3x slower.
    # 0.4.0 is the release that added the BACKWARD (dQ/dK/dV and dBias) for
    # both sm_70 and sm_75, so a design run on a T4 keeps the kernel too.
    _sh('pip install -q colabfold-legacy-kernels==0.4.0 colabfold-kernels==0.4.0',
        'installing the pre-Ampere fused kernels')
  elif _cc0 is not None:
    # ... and its sibling for Ampere and newer. Pure Python (Pallas), so this
    # is a small wheel and no build. It matters most on the L4 Colab offers:
    # tokamax's Triton refuses an Ada card outright ('Not supported on NVIDIA
    # A10') and cuDNN is what we fell back to -- this measured 3.3x cuDNN and
    # 11.9x XLA on an A10 at this model's triangle-attention shape.
    _sh('pip install -q colabfold-kernels==0.4.0',
        'installing the Ampere+ fused kernels')
  # zstandard IS one of them -- params.py, post_processing.py and
  # folding_input.py all import it. It happened to be preinstalled on the
  # GPU images, so its absence only showed up on a TPU runtime, as
  # `ModuleNotFoundError: No module named zstandard` from inside the fold,
  # long after the install cell had reported success.
  _sh("pip install -q dm-haiku==0.0.17 rdkit==2025.9.4 "
      "tokamax==0.0.11 ml_collections zstandard", 'installing dependencies')
  if IS_AF2:
    os.system("apt-get -qq install -y aria2 > /dev/null 2>&1")  # AF2's tar is 5.3 GB
  # Retried: PyPI's index can lag a just-published release by a few minutes.
  for _try in range(4):
    if os.system(f'pip install -q --no-deps alphafold3-colabfold=={VERSION}') == 0:
      break
    print(f'pip could not find {VERSION} yet; retrying in 20 s')
    time.sleep(20)
  else:
    raise RuntimeError(f'could not install alphafold3-colabfold=={VERSION}')

  # haiku 0.0.17 still calls the moved jax.core.DropVar.
  os.system("sed -i 's/jax.core.DropVar/jax.extend.core.DropVar/g' /usr/local/lib/python*/dist-packages/haiku/_src/jaxpr_info.py")
  import alphafold3  # confirms the install before anything depends on it
  os.system('touch ALPHAFOLD3_READY')
  print(f'Packages installed ({alphafold3.__file__}).')

# py2Dmol IS ASKED FOR BY IMPORTING, NOT BY THE READY FLAG. The pip block
# above only runs on a fresh machine, so a session that installed before
# this line existed has the flag and no viewer, and the display cells die on
# `import py2Dmol`. An import is cheap on a warm machine and self-healing on
# a stale one -- the same reasoning as the overlay note below.
try:
  import py2Dmol  # noqa: F401
except ImportError:
  _sh('pip install -q git+https://github.com/sokrypton/py2Dmol.git',
      'installing py2Dmol')   # the wheel on PyPI lags the repo

# THE BRANCH OVERLAY RUNS EVERY TIME, not only on a fresh install. It used to
# sit inside the `ALPHAFOLD3_READY` guard, so a session that had already
# installed never refreshed it -- and a file ADDED on the branch (staged.py)
# could never arrive at all: `ImportError: cannot import name staged`, on a
# notebook whose overlay list had already been fixed. Five wgets, so there is
# no reason to skip them.
# EXPERIMENTAL BRANCH OVERLAY. The live cell needs library changes that are
# not in any release, so the branch's Python is copied over the installed
# wheel. That is legitimate here and nowhere else: `colab` changes no C++, so
# the compiled extension in the wheel is exactly the one this source expects.
# The moment a .cc changes on the branch this stops being true, which is what
# the assertion below is for -- and then the install becomes
#     pip install git+https://github.com/sokrypton/alphafold3@colab
# which builds the extension from scratch (~10 minutes on Colab).
# run_alphafold.py is a top-level script, not part of the package, and it is
# fetched HERE rather than under the install guard for the same reason as the
# rest of the overlay: on a warm session the guard is skipped and a stale copy
# would survive a push.
# run_alphafold.py COMES OUT OF THE SAME TARBALL as the package, below.
#
# It used to be its own wget from raw.githubusercontent, and that host serves a
# stale blob for a long time after a push -- a cache-buster query string did
# NOT help (verified: a fresh wget on the VM returned a file missing the
# newest commit while the GitHub API reported that commit as the branch head).
# Two failures came of it, and both looked like code bugs: the fold died on a
# validation the branch no longer had, quoting an error message that no longer
# existed. It is also the same shape as the overlay-list bug -- the script and
# the package coming from different places and disagreeing. One snapshot, one
# fetch, no way for them to drift.

# ONE TARBALL, for the script and (on a branch) the package. A tag needs
# run_alphafold.py too -- it is not part of the wheel -- so this runs either
# way and only the package overlay is conditional.
import importlib.metadata as _md
_root = os.path.dirname(_md.distribution('alphafold3-colabfold')
                        .locate_file('alphafold3'))
_ref = ('refs/heads/' + SOURCE if SOURCE != f'v{VERSION}'
        else 'refs/tags/' + SOURCE)
_sh(f'wget -q -O branch.tar.gz https://codeload.github.com'
    f'/sokrypton/alphafold3/tar.gz/{_ref}?nocache={int(time.time())}',
    f'fetching {SOURCE}')
_sh('rm -rf branch_src && mkdir branch_src && '
    'tar xzf branch.tar.gz -C branch_src --strip-components=1', 'unpacking it')
_sh('cp branch_src/run_alphafold.py run_alphafold.py',
    'taking run_alphafold.py from the tarball')

if SOURCE != f'v{VERSION}':
  # THE WHOLE TREE, not a list of files. There WAS a list, fetched from the
  # branch so it could not go stale against the branch -- and it went stale
  # against the WHEEL instead, which is the comparison that actually matters:
  # `git diff main colab` is empty for a file that main changed AFTER the
  # release was cut, so evoformer.py stayed at 3.1.11 while run_alphafold.py
  # came from the branch and asked it for bfloat16='intermediate'. The wheel's
  # assert had never heard of that value and every fold died in the first
  # recycle. A tarball of the branch has no list to keep in step: 8 MB, about
  # a second, one request instead of six.
  # `.` copies the CONTENTS, merging into the installed package, so the
  # compiled extension (.so) stays where the wheel put it -- which is the
  # whole reason overlaying source on a wheel is legitimate here.
  _sh(f'cp -r branch_src/src/alphafold3/. {_root}/alphafold3/',
      'overlaying the package')
  _sh('cp branch_src/dev/live/live_frames.py live_frames.py',
      'fetching live_frames.py')
else:
  # live_frames.py lives in dev/ and is only overlaid from a branch, so on a
  # release SOURCE the live cell has nothing to import. Say which cell and
  # why, rather than leaving it a bare ModuleNotFoundError there.
  print(f'note: SOURCE={SOURCE} is a release, so live_view in the run cell'
        ' is not available -- set SOURCE to a branch to use it.')
  print(f'overlaid the {SOURCE} branch onto the {VERSION} wheel')

if SOURCE != f'v{VERSION}':
  # Proves the overlay landed rather than trusting the download. Checked by
  # READING the files: importing run_alphafold here would pull in
  # alphafold3.constants, whose pickles the input cell has not written yet
  # (FileNotFoundError: chemical_component_sets.pickle), and the failure
  # would land before CACHE_DIR is even defined.
  import importlib.metadata as _md2
  _r = os.path.dirname(_md2.distribution('alphafold3-colabfold')
                       .locate_file('alphafold3'))
  for _f, _needle in (
      ('run_alphafold.py', 'def live_model'),
      (os.path.join(_r, 'alphafold3/model/model.py'), "stage='all'"),
      (os.path.join(_r, 'alphafold3/model/staged.py'), 'def make_stages'),
  ):
    assert _needle in open(_f).read(), (
        f'{_f} is not the {SOURCE} version ({_needle!r} missing) -- the '
        'overlay did not take effect and the live cell would fail')

# tokamax's Triton kernels need more shared memory than Ada cards have, so
# restrict them to datacenter GPUs (A100 cc 8.0, H100 cc 9.0+).
#
# DO NOT IMPORT tokamax TO DO IT, AND DO NOT DO IT OFF A GPU. `import tokamax`
# imports jax, and on a TPU runtime the first process to touch jax OWNS the
# chip -- so this line, whose only job is to edit a CUDA policy, took the TPU
# and left the fold's own subprocess with
#     ABORTED: The TPU is already in use by process with pid <the kernel>
# and a silent fall back to "CPU-only inference". find_spec locates the file
# without executing the package, and the patch is skipped where it means
# nothing. (platform.detect_device() is jax-free by design -- checked.)
from alphafold3.model.components import platform as _plat
_dev0, _cap0 = _plat.detect_device()
if _dev0 != 'gpu' or not _plat.needs_tokamax_patch(_cap0):
  print(f'tokamax patch not needed on this device ({_dev0}, cc {_cap0}).')
try:
  import importlib.util as _ilu
  if _dev0 != 'gpu' or not _plat.needs_tokamax_patch(_cap0):
    raise SystemExit  # caught below; nothing to patch
  _spec = _ilu.find_spec('tokamax')
  _gu = os.path.join(os.path.dirname(_spec.origin), '_src', 'gpu_utils.py')
  _s = open(_gu).read()
  _old = 'return float(device.compute_capability) >= 8.0'
  _new = ('cc = float(device.compute_capability)\n'
          '  return cc == 8.0 or cc >= 9.0  # datacenter only; Ada/L4 lack shared memory')
  if _old in _s:
    open(_gu, 'w').write(_s.replace(_old, _new))
    print('Patched tokamax: Triton restricted to datacenter GPUs (L4/Ada -> XLA).')
except SystemExit:
  pass
except Exception as _e:
  print(f'(tokamax patch skipped: {_e})')

# Weights, fetched in the background by the same code the run uses.
STAMP = f'WEIGHTS_DONE_{model}_{PRECISION}'
if IS_AF3 and not os.path.isfile(STAMP):
  # DeepMind's own release, subject to the AlphaFold 3 terms of use, which
  # run_alphafold prints at startup. A copy you already have in NATIVE_DIR is
  # used as-is.
  os.makedirs(NATIVE_DIR, exist_ok=True)
  if glob.glob(f'{NATIVE_DIR}/*.bin.zst'):
    open(STAMP, 'w').close()
    print(f'Using the AlphaFold 3 parameters already in {NATIVE_DIR}/.')
  else:
    print('Downloading AlphaFold 3 parameters (~1 GB)...')
    os.system(f'(wget -O {NATIVE_DIR}/af3.bin.zst "{AF3_WEIGHTS_URL}"'
              f' > {STAMP}.log 2>&1 && touch {STAMP}) &')
elif not (IS_AF3 or os.path.isfile(STAMP)):
  _script, _args = ('prefetch_af2.py', AF2_DIR) if IS_AF2 else (
      'prefetch_weights.py', f'{model} {PRECISION}')
  print(f'Downloading {"official AlphaFold 2 parameters (CC BY 4.0)" if IS_AF2 else model} weights...')
  with open(_script, 'w') as fh:
    fh.write('import sys\n'
             'from alphafold3.model import weights\n'
             + ('print(weights.ensure_af2_params(sys.argv[1]))\n' if IS_AF2 else
                'print(weights.ensure_weights(sys.argv[1], None, precision=sys.argv[2]))\n'))
  os.system(f'(python {_script} {_args} > {STAMP}.log 2>&1 && touch {STAMP}) &')

# /tmp is wiped with the VM, so a fresh session recompiles (~53 s); Drive survives.
CACHE_DIR = '/tmp/af3_cache'
if persist_cache_to_drive:
  try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = '/content/drive/MyDrive/.af3_cache'
    os.makedirs(CACHE_DIR, exist_ok=True)
    print(f'Compile cache: {CACHE_DIR} (survives this session)')
  except Exception as _e:
    print(f'(Drive mount failed, using {CACHE_DIR}: {_e})')


def _await(sentinel, limit=1200):
  """Wait for a background job, reporting its log if it never finishes."""
  t0 = time.time()
  while not os.path.isfile(sentinel):
    if time.time() - t0 > limit:
      log = f'{sentinel}.log'
      tail = open(log).read()[-1500:] if os.path.isfile(log) else '(no output captured)'
      raise RuntimeError(f'{sentinel} did not appear within {limit} s. '
                         f'Tail of {log}:\n{tail}')
    time.sleep(5)
  print(f'{sentinel} \u2713  ({time.time() - t0:.0f} s)')


_await(STAMP)

if IS_AF3 and os.path.getsize(f'{NATIVE_DIR}/af3.bin.zst') < 1_000_000:
  raise RuntimeError('the AlphaFold 3 download is incomplete - re-run this cell.')

print(f'Setup complete!  Model: {model}.')
if model == 'chai1':
  print('NOTE: chai-1 is running WITHOUT ESM2 embeddings, which are most of its token\n'
        '      features. Expect worse structures than chai-lab itself produces.')
print(f'Setup took {time.time() - _T0:.0f} s.')


In [ ]:
#@title Input sequences
import re, os, json, hashlib

#@markdown ### Molecules
#@markdown Separate multiple chains within a box using `:` (extra colons are fine: `A::::B` == `A:B`). Leave a box empty if unused; full details in the Instructions cell.
protein = 'PIAQIHILEGRSDEQKETLIREVSEAISRSLDAPLTSVRVIITEMAKGHFGIGGELASK' #@param {type:"string"}
dna = '' #@param {type:"string"}
rna = '' #@param {type:"string"}
ligand_ccd = '' #@param {type:"string"}
ligand_smiles = '' #@param {type:"string"}

#@markdown ### Run settings
jobname = 'test' #@param {type:"string"}
msa_mode = "mmseqs2_server" #@param ["mmseqs2_server", "single_sequence"]
seeds = '1' #@param {type:"string"}
on_existing = "overwrite" #@param ["overwrite", "skip"]
#@markdown - `msa_mode`: `single_sequence` skips the MSA (faster, lower accuracy).
#@markdown - `seeds`: comma-separated, e.g. `1,2,3`.
#@markdown - `on_existing`: `overwrite` replaces this job's previous results; `skip` keeps them.

# Headless overrides -- see the install cell.
import json as _json, os as _os
for _k, _v in _json.loads(_os.environ.get('AF3_NB_OVERRIDES', '{}')).items():
  if _k in globals():
    globals()[_k] = _v
    print(f'override: {_k} = {_v!r}')

# Split a box into entries: collapse colon runs, drop whitespace, skip empties
def split_entries(s):
  s = re.sub(r':+', ':', s).strip(':')
  return [e for e in (''.join(tok.split()) for tok in s.split(':')) if e]

prot_seqs   = [e.upper() for e in split_entries(protein)]
dna_seqs    = [e.upper() for e in split_entries(dna)]
rna_seqs    = [e.upper() for e in split_entries(rna)]
ccd_codes   = [e.upper() for e in split_entries(ligand_ccd)]
smiles_strs = split_entries(ligand_smiles)         # case-sensitive: leave as typed

# Fetch chemical definitions for the components this input names, from
# files.rcsb.org (~0.6 s). A code that is not fetched raises when folding.
with open('prefetch_ccd.py', 'w') as fh:
  fh.write('import sys, os, importlib.metadata as md\n'
           'from alphafold3.constants import ccd_fetch\n'
           'root = os.path.dirname(md.distribution("alphafold3-colabfold")'
           '.locate_file("alphafold3"))\n'
           'conv = os.path.join(root, "alphafold3", "constants", "converters")\n'
           'os.makedirs(conv, exist_ok=True)\n'
           'ccd_fetch.write_pickles(ccd_fetch.codes_for_input(extra=sys.argv[1:]),\n'
           '  os.path.join(conv, "ccd.pickle"),\n'
           '  os.path.join(conv, "chemical_component_sets.pickle"),\n'
           '  libcifpp_dir=os.path.join(root, "share", "libcifpp"))\n')
print(f'Fetching the CCD: 35 standard residues'
      + (f' + {", ".join(ccd_codes)}' if ccd_codes else '') + ' ...')
if os.system('python prefetch_ccd.py ' + ' '.join(ccd_codes)) != 0:
  raise RuntimeError('could not build the CCD tables; see the output above')

# Build AF3 chain entities (IDs A, B, C, ... in canonical order)
CHAIN_IDS = list('ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz')
chains, prot_groups, idx = [], {}, 0

for seq in prot_seqs:
  cid = CHAIN_IDS[idx]; idx += 1
  if seq in prot_groups:                           # merge identical seqs -> homo-oligomer
    ent = prot_groups[seq]
    ids = ent['id'] if isinstance(ent['id'], list) else [ent['id']]
    ent['id'] = ids + [cid]
  else:
    ent = {'id': cid, 'sequence': seq, 'templates': []}
    if msa_mode == 'single_sequence':
      ent.update({'unpairedMsa': f'>query\n{seq}\n', 'pairedMsa': ''})
    prot_groups[seq] = ent
    chains.append({'protein': ent})

for seq in rna_seqs:
  c = {'id': CHAIN_IDS[idx], 'sequence': seq}
  if msa_mode == 'single_sequence':
    c['unpairedMsa'] = f'>query\n{seq}\n'
  chains.append({'rna': c}); idx += 1

for seq in dna_seqs:
  chains.append({'dna': {'id': CHAIN_IDS[idx], 'sequence': seq}}); idx += 1

for code in ccd_codes:
  chains.append({'ligand': {'id': CHAIN_IDS[idx], 'ccdCodes': [code]}}); idx += 1

for smiles in smiles_strs:
  chains.append({'ligand': {'id': CHAIN_IDS[idx], 'smiles': smiles}}); idx += 1

if not chains:
  raise ValueError('No valid input found - fill in at least one box.')

# Seeds: pull out integers regardless of separators, dedupe, default to [1]
seed_list = []
for tok in re.findall(r'\d+', seeds):
  v = int(tok)
  if v not in seed_list:
    seed_list.append(v)
if not seed_list:
  seed_list = [1]

# Deterministic, lower-cased job name from inputs+seeds.
# Same input+seeds -> same folder (so re-runs reuse it instead of piling up).
# Lower-cased to match run_alphafold.py's sanitised_name() output directory.
def _flat(mol):
  if 'sequence' in mol: return mol['sequence']
  if 'ccdCodes' in mol: return ','.join(mol['ccdCodes'])
  return mol.get('smiles', '?')
flat = ':'.join(_flat(list(c.values())[0]) for c in chains) + '|seeds=' + ','.join(map(str, seed_list))
basejob = (re.sub(r'\W+', '', ''.join(jobname.split())) or 'job').lower()
jobname = basejob + '_' + hashlib.sha1(flat.encode()).hexdigest()[:5]

# Input JSON goes to a temp dir; ALL results land in ONE folder: af3_output/<jobname>/
INPUT_DIR  = '/tmp/af3_inputs'
OUTPUT_DIR = 'af3_output'
job_dir    = f'{OUTPUT_DIR}/{jobname}'

fold_input = {
    'name': jobname,
    'sequences': chains,
    'modelSeeds': seed_list,
    'dialect': 'alphafold3',
    'version': 1,
}
os.makedirs(INPUT_DIR, exist_ok=True)
json_path = f'{INPUT_DIR}/{jobname}.json'
with open(json_path, 'w') as f:
  json.dump(fold_input, f, indent=2)

print(f'Job "{jobname}"  ->  results will be written to {job_dir}/')
fold_input


In [ ]:
#@title Run the model
#@markdown ### Either family
num_recycles = 3 #@param {type:"integer"}
num_msa = 512 #@param [512, 1024, 256, 128, 64, 32, 1, 0] {type:"raw"}
live_view = True #@param {type:"boolean"}

#@markdown ### AlphaFold 3 family
num_diffusion_samples = 5 #@param {type:"integer"}
diffusion_steps = 32 #@param {type:"integer"}

#@markdown ### AlphaFold 2
af2_num_models = 1 #@param [1, 2, 3, 4, 5] {type:"raw"}

import os, shutil, subprocess, glob, time
_T0 = time.time()

# Headless overrides -- see the install cell.
import json as _json, os as _os
for _k, _v in _json.loads(_os.environ.get('AF3_NB_OVERRIDES', '{}')).items():
  if _k in globals():
    globals()[_k] = _v
    print(f'override: {_k} = {_v!r}')

# 0 SURVIVES: it means the model's own default, and max(1, ...) would have
# turned it into a single recycle -- a quiet, much worse fold.
num_recycles = max(0, int(num_recycles))
num_diffusion_samples = max(1, int(num_diffusion_samples))

# ATTENTION KERNEL AND XLA FLAGS, FROM THE LIBRARY, NOT FROM THIS CELL.
# This block used to carry its own copy of the device matrix, and it had gone
# wrong in a way no test could see: it sent Ada and consumer Ampere (L4, A10,
# RTX 30/40) down the XLA attention path because those cards cannot launch the
# Triton kernels. They cannot -- but they run cuDNN, which is 2.64x on triangle
# attention at 384 tokens (11.89 ms -> 4.50 ms, max|d| 0.0000) and 20% off a
# whole fold. alphafold3.model.components.platform has said so all along.
# It imports in 40 ms and does not pull in jax.
try:
  from alphafold3.model.components import platform as _platform
  _dev = _platform.attention_config()
  flash_impl, nojit, xla_flags = _dev['attention'], _dev['nojit'], _dev['xla_flags']
  # The full sentence explaining WHY this card gets this kernel is
  # _dev['reason'], left in globals rather than printed: it is a paragraph,
  # it is the same paragraph every run, and it is only read when something
  # is wrong. One line here, the reason on request.
except Exception as _e:
  # Never let a fold fail because the probe did: XLA runs everywhere.
  flash_impl, nojit, xla_flags = 'xla', False, []
  print(f'(device probe failed, falling back to XLA attention: {_e})')

# Export the XLA flags so the child shell -- and JAX inside it -- inherit them.
cur = os.environ.get('XLA_FLAGS', '')
for f in xla_flags:
  if f not in cur:
    cur = (cur + ' ' + f).strip()
if cur:
  os.environ['XLA_FLAGS'] = cur

# ONE LINE, not five. The device probe, the flags and the model were a
# paragraph each before anything had happened; `_dev['reason']` and
# os.environ['XLA_FLAGS'] are both still there to print if a fold misbehaves.
print(f"{model} | attention: {flash_impl}"
      + (f" | {len(xla_flags)} XLA flag(s)" if xla_flags else ""))

# WHICH PATH RAN. The AF2 live view is new and has no test behind it, so a
# failure there must not strand anyone: it says what went wrong and the
# normal run happens below, which is what an untested path is allowed to
# cost. Loud, not silent -- a fallback nobody is told about is how a fold
# quietly stops being the fold you asked for.
_folded = False

# ---- live, AlphaFold 2: one frame per RECYCLE ----
# AF2 has no diffusion, but it has recycles, and a recycle is its only
# intermediate structure. The loop they live in is Python (AF2Runner.apply;
# RunModel.apply is jitted over ONE pass), so watching them costs no extra
# compile -- alphafold3.af2 takes an `on_recycle` callback for exactly this.
# Everything after the fold is the normal AF2 path, so the outputs are the
# ones a subprocess run would write.
if live_view and IS_AF2:
 try:
  import gc, numpy as np, jax
  from tqdm.auto import tqdm
  import py2Dmol
  from absl import flags
  import run_alphafold as RA

  if not flags.FLAGS.is_parsed():
    flags.FLAGS(['run_alphafold.py', '--norun_data_pipeline',
                 f'--cache_dir={CACHE_DIR}'])
  flags.FLAGS.model = model
  flags.FLAGS.force_output_dir = True
  _jax_cache = os.path.join(CACHE_DIR, 'jax')
  os.makedirs(_jax_cache, exist_ok=True)
  jax.config.update('jax_compilation_cache_dir', _jax_cache)

  from alphafold3.common import folding_input
  from alphafold3.af2 import inference as af2_inference
  from alphafold3.af2.output import atom37_to_token_atoms
  from alphafold3.model import model_registry as _reg
  import live_frames as LF

  fi = folding_input.load_fold_inputs_from_path(json_path).__next__()
  if msa_mode == 'mmseqs2_server' and any(
      c.unpaired_msa is None for c in fi.protein_chains):
    from alphafold3.data import msa_server
    fi = msa_server.fill_missing_msas(fi)

  # The CLI's own construction (the spec.engine == 'af2' branch): from the
  # registry's spec, and NOT ModelRunner.
  # Each 0 field is OMITTED rather than passed, so the runner's own defaults
  # apply -- 3 recycles, 512 clustered MSA rows, 1024 extra.
  _af2_kw = {}
  if int(num_recycles):
    _af2_kw['num_recycles'] = int(num_recycles)
  if int(num_msa):
    _af2_kw['num_msa'] = int(num_msa)
    # AF2's extra stack at twice the clustered rows: its own defaults are 512
    # and 1024, so this keeps the ratio the model was configured with rather
    # than asking a reader to hold two numbers.
    _af2_kw['num_extra_msa'] = int(num_msa) * 2
  runner = af2_inference.AF2ModelRunner(
      _reg.get(model), device=jax.local_devices()[0], model_dir=AF2_DIR,
      use_templates=False, **_af2_kw)
  # ...and the bar needs the number that will actually run, which only the
  # runner knows once the default has been applied.
  _nrec = int(getattr(runner, '_num_recycles', int(num_recycles) or 3))

  viewer = py2Dmol.view(size=(420, 420), style='cartoon')
  viewer.show()
  bar = tqdm(total=_nrec + 1, unit='pass', desc='recycle', leave=True)
  _bobj = [None]

  # A RECYCLE IS AF2's ONLY INTERMEDIATE STRUCTURE, and its loop is Python
  # (AF2Runner.apply; RunModel.apply is jitted over one pass), so the hook
  # costs no extra compile. This is the same shape as the AF3 branch's
  # on_frame, one axis shorter: no diffusion to step through.
  def on_recycle(i, out):
    bar.set_description(f'recycle {i + 1}/{_nrec + 1}')
    bar.update(1)
    if _bobj[0] is None:
      return
    atom37 = np.asarray(out['structure_module']['final_atom_positions'])
    coords, _ = atom37_to_token_atoms(atom37, _bobj[0])
    xyz, chains, resids = LF.frame_positions(coords, _bobj[0])
    viewer.add(xyz, chains=chains, residue_numbers=resids, name='fold')

  # THE CALLBACK RIDES ON THE RUNNER, which is what lets the CLI's pipeline
  # drive an animated fold: process_fold_input calls run_inference, so
  # binding the hook here needs no flag and no second code path. The batch
  # is taken on the way past for the same reason the AF3 branch takes it.
  _orig = runner.run_inference
  def _run_inference(featurised_example, *a, **k):
    if _bobj[0] is None:
      _bobj[0] = LF.as_batch(featurised_example)
    k.setdefault('on_recycle', on_recycle)
    return _orig(featurised_example, *a, **k)
  runner.run_inference = _run_inference

  _t0 = time.time()
  shutil.rmtree(job_dir, ignore_errors=True)
  RA.process_fold_input(
      fold_input=fi, data_pipeline_config=None, model_runner=runner,
      output_dir=job_dir, buckets=None, force_output_dir=True)
  bar.close()
  print(f'done in {time.time() - _t0:.0f}s -> {job_dir}/'
        '  (drag the slider to replay the recycles)')
  _folded = True
 except Exception as _live_err:
  print(f'the AF2 live view failed ({type(_live_err).__name__}: {_live_err});'
        ' running the normal way instead')

if not _folded and live_view and not IS_AF2:
 try:
  import gc, numpy as np, jax
  from tqdm.auto import tqdm
  import py2Dmol
  from absl import flags
  import run_alphafold as RA

  # FREE THE PREVIOUS MODEL FIRST. This branch keeps parameters and compiled
  # executables in THIS process, unlike the subprocess below which hands
  # everything back when it exits. Re-running with a different model without
  # this holds two sets of weights on a 16 GB card.
  for _stale in ('runner', 'viewer', 'frames'):
    if _stale in globals():
      del globals()[_stale]
  gc.collect()
  try:
    jax.clear_caches()
  except Exception:
    pass

  if not flags.FLAGS.is_parsed():
    flags.FLAGS(['run_alphafold.py', '--norun_data_pipeline',
                 f'--cache_dir={CACHE_DIR}'])
  flags.FLAGS.model = model
  flags.FLAGS.flash_attention_implementation = flash_impl
  # ONE TRUNK PASS, CALLED PER RECYCLE, which is what makes the frames exist:
  # run_alphafold's own live driver rather than a second copy of it here.
  flags.FLAGS.stepwise_recycles = True
  flags.FLAGS.force_output_dir = True
  # The compile cache, which nothing in-process sets: the CLI does it in
  # main(), and main() is not what runs here.
  _jax_cache = os.path.join(CACHE_DIR, 'jax')
  os.makedirs(_jax_cache, exist_ok=True)
  jax.config.update('jax_compilation_cache_dir', _jax_cache)

  from alphafold3.common import folding_input
  from alphafold3.model import weights as _w
  import live_frames as LF

  fi = folding_input.load_fold_inputs_from_path(json_path).__next__()
  # The MSA, which --norun_data_pipeline means nobody else will fetch. The CLI
  # does this in main() too, for the same reason.
  if msa_mode == 'mmseqs2_server' and any(
      c.unpaired_msa is None for c in fi.protein_chains):
    from alphafold3.data import msa_server
    fi = msa_server.fill_missing_msas(fi)

  # Omitted rather than passed when 0, so make_model_config's own 10 applies.
  _cfg_kw = {'num_recycles': int(num_recycles)} if int(num_recycles) else {}
  cfg = RA.make_model_config(
      model_name=model, num_diffusion_samples=num_diffusion_samples,
      flash_attention_implementation=flash_impl, **_cfg_kw)
  if int(num_msa):
    cfg.evoformer.num_msa = int(num_msa)
  # ...and the counts the bar and the frame tests need are READ BACK, because
  # with 0 the notebook does not know them and the config does.
  _nrec = int(cfg.num_recycles)
  if int(diffusion_steps):
    cfg.heads.diffusion.eval.steps = int(diffusion_steps)
  _nsteps = int(cfg.heads.diffusion.eval.steps)
  cfg.heads.diffusion.eval.stepwise = True    # frames per denoise step
  weights_dir = (NATIVE_DIR if IS_AF3 else _w.default_dir(model, PRECISION))
  runner = RA.ModelRunner(config=cfg, device=jax.local_devices()[0],
                          model_dir=weights_dir)

  # ONE bar for the whole fold: the phases are very different lengths and each
  # compiles the first time it runs, so what stops the cell looking hung is
  # the phase NAME beside a bar that never restarts.
  bar = tqdm(total=(_nrec + 1) + _nsteps + 1, unit='step', leave=True,
             desc='trunk',
             bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]')
  viewer = py2Dmol.view(size=(420, 420), style='cartoon', heatmap='tab')
  viewer.show()

  _bobj = [None]          # the Batch the coordinate helpers want, once known
  prev = [None]
  # EVERY STEP, ONE STEP LATE. np.asarray on a device array waits for that
  # step, so converting frame t inside step t drains the pipeline. This starts
  # the host copy for t and emits t-1, whose copy has had a whole step to land.
  _queued = []
  def _emit(arr):
    xyz, chains, resids = LF.frame_positions(np.asarray(arr), _bobj[0])
    viewer.add(xyz, chains=chains, residue_numbers=resids, name='fold')
  def _stream(arr):
    try:
      arr.copy_to_host_async()
    except AttributeError:
      pass
    _queued.append(arr)
    while len(_queued) > 1:
      _emit(_queued.pop(0))

  def on_frame(kind, i, data):
    if kind == 'recycle':
      cm = np.asarray(LF.contact_map(data['contacts']), dtype=np.float32)
      moved = LF.recycle_distance(prev[0], cm); prev[0] = cm
      bar.set_description(f'trunk pass {i + 1}/{_nrec + 1}')
      bar.set_postfix_str('' if not i else f'contacts moved {moved:.4f}')
      bar.update(1)
      # An empty (0, 3) carries the map without claiming any coordinates.
      viewer.add(np.zeros((0, 3), dtype=np.float32), name='trunk',
                 maps={'contact': cm})
      if i == _nrec:
        bar.set_description('diffusion')
        bar.set_postfix_str('')
    elif _bobj[0] is not None:
      _stream(data[0] if getattr(data, 'ndim', 0) == 4 else data)
    if kind == 'diffusion':
      bar.set_description(f'diffusion {i + 1}/{_nsteps}')
      bar.update(1)
      if i == _nsteps - 1:
        bar.set_description('scoring (pLDDT, PAE)')

  # THE BATCH THE FRAMES ARE DRAWN AGAINST. process_fold_input featurises
  # internally, so the notebook never sees the example -- and frame_positions
  # needs it. run_inference is where it arrives, so it is taken there, once.
  _orig_run_inference = runner.run_inference
  def _run_inference(featurised_example, *a, **k):
    if _bobj[0] is None:
      _bobj[0] = LF.as_batch(featurised_example)
    return _orig_run_inference(featurised_example, *a, **k)
  runner.run_inference = _run_inference

  RA._FRAME_CALLBACK[0] = on_frame
  _t0 = time.time()
  try:
    shutil.rmtree(job_dir, ignore_errors=True)
    # THE CLI'S OWN PIPELINE, not a second copy of it: featurisation, every
    # seed, every sample, the ranking, the confidences and write_outputs.
    RA.process_fold_input(
        fold_input=fi, data_pipeline_config=None, model_runner=runner,
        output_dir=job_dir, buckets=None, force_output_dir=True)
  finally:
    RA._FRAME_CALLBACK[0] = None
  _drain = _queued
  while _drain:
    _emit(_drain.pop(0))
  bar.update(1)
  bar.set_description('done')
  bar.close()
  print(f'done in {time.time() - _t0:.0f}s -> {job_dir}/'
        '  (drag the slider to replay the fold)')
  _folded = True
 except Exception as _live_err:
  print(f'the live view failed ({type(_live_err).__name__}: {_live_err});'
        ' running the normal way instead')

# ---- standard: one fused graph in a subprocess ----
if not _folded:
  os.makedirs(OUTPUT_DIR, exist_ok=True)

  # Re-run policy (one folder per job, no timestamped duplicates):
  #   overwrite -> wipe this job's folder and recompute
  #   skip      -> if a finished result (.cif) is already there, don't recompute
  have_results = os.path.isdir(job_dir) and any(f.endswith('.cif') for f in os.listdir(job_dir))
  run_it = not (on_existing == 'skip' and have_results)
  if run_it:
    shutil.rmtree(job_dir, ignore_errors=True)   # start clean so exactly one folder is produced

  cmd = [
      'python', 'run_alphafold.py',
      f'--json_path={json_path}',
      f'--model={model}',
      '--norun_data_pipeline',
      f'--output_dir={OUTPUT_DIR}',
      f'--cache_dir={CACHE_DIR}',
      # THE TRACE AND LOWERING, which the compile cache does NOT store. This
      # cell runs the fold in a FRESH PROCESS every time, so that floor is the
      # user's wait: measured by folding one input under two seeds in one
      # process (the second retraces nothing) -- 24.22 s then 13.47 s, i.e.
      # 10.8 s / 44% of a first fold is trace + lowering. --lowercache_dir
      # serialises the executable itself (3.1 MB, bitwise-identical on reload)
      # and took that fold to 18.6 s, -23%. It lives under CACHE_DIR, so
      # persist_cache_to_drive carries it between sessions like the rest.
      # One cost, once: if you already have a Drive cache from before this
      # line existed, the first fold has to lower again under a different key.
      f'--lowercache_dir={CACHE_DIR}/lower',
      # ONE EXECUTABLE FOR EVERY RECYCLE COUNT. The count is otherwise a
      # Python int baked into the graph, so a cache built at 10 misses at 3
      # and the user pays a full compile for moving a form field. Traced, it
      # is bitwise identical (max|d| 0.000000 on all 5 samples of 1STP) and
      # measured 3.8% FASTER on an A10 -- 18.36 s against 17.66 s, n=3.
      # Prediction only: a fori_loop with a dynamic bound is not
      # reverse-differentiable, so the design path must not set it.
      '--dynamic_recycles',
      '--force_output_dir',          # reuse af3_output/<jobname>/ instead of a timestamped copy
      f'--flash_attention_implementation={flash_impl}',
      *( [f'--num_recycles={int(num_recycles)}'] if int(num_recycles) else [] ),
      f'--num_diffusion_samples={num_diffusion_samples}',
      *( [f'--num_msa={int(num_msa)}',
          # AF2's extra stack, at its default ratio to the clustered one.
          # Ignored by every AF3-family model, which has one MSA stack.
          f'--num_extra_msa={int(num_msa) * 2}'] if int(num_msa) else [] ),
      *( [f'--num_sampling_steps={int(diffusion_steps)}']
         if int(diffusion_steps) else [] ),
      f'--af2_num_models={int(af2_num_models)}',
  ]
  if msa_mode == 'mmseqs2_server':
    cmd.append('--use_msa_server')
  # chai-1 and ESMFold2 fold from a language model, downloaded on first use.
  # Without it they are a different model, not a slightly worse one.
  if model == 'chai1' or model.startswith('esmfold2'):
    cmd.append('--use_esm_embeddings')
  if nojit:
    cmd.append('--nojit')
  if IS_AF3 or IS_AF2:
    cmd.append(f'--model_dir={AF2_DIR if IS_AF2 else NATIVE_DIR}')

  cmd = ' '.join(cmd)
  if run_it:
    print(cmd)
    # Popen rather than `!`: streams the output and gives an exit status.
    _p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1)
    for _line in _p.stdout:
      print(_line, end='')
    _rc = _p.wait()
    _cifs = glob.glob(f'{job_dir}/**/*.cif', recursive=True)
    if _rc != 0 or not _cifs:
      raise RuntimeError(
          f'the fold FAILED (exit {_rc}, {len(_cifs)} structures written). '
          'The output above is the whole story; scroll up for the error.')
    print(f'\nDone -> {job_dir}/  ({len(_cifs)} structures, '
          f'{time.time() - _T0:.0f} s)')
  else:
    print(f'Skipping: results already exist in {job_dir}/  (set on_existing=overwrite to recompute).')


In [ ]:
#@title Display structures + PAE (py2Dmol)
import csv, glob, os, json
import numpy as np
import py2Dmol

load_as_frames = True #@param {type:"boolean"}
viewer_size = 400
#@markdown All models load together, best first. `load_as_frames` on makes them
#@markdown frames to play through, off gives a dropdown. Drag a box on the PAE
#@markdown matrix to highlight residues.

# All models in rank order (best first): from the ranking CSV, fall back to globbing.
def collect_models():
  ranking_csv = f'{job_dir}/{jobname}_ranking_scores.csv'
  cifs = []
  if os.path.exists(ranking_csv):
    rows = []
    with open(ranking_csv) as f:
      for r in csv.DictReader(f):
        rows.append((float(r['ranking_score']), int(r['seed']), int(r['sample'])))
    for _, seed, sample in sorted(rows, reverse=True):
      d = f'{job_dir}/seed-{seed}_sample-{sample}'
      hit = sorted(glob.glob(f'{d}/*_model.cif')) or sorted(glob.glob(f'{d}/*.cif'))
      if hit:
        cifs.append(hit[0])
  if not cifs:
    cifs = (sorted(glob.glob(f'{job_dir}/**/*_model.cif', recursive=True))
            or sorted(glob.glob(f'{job_dir}/**/*.cif', recursive=True)))
  return cifs

# Per-model PAE: confidences.json next to the CIF, else the top-level one.
def load_pae(cif):
  d = os.path.dirname(cif)
  cands = [p for p in glob.glob(f'{d}/*_confidences.json')
           if 'summary' not in os.path.basename(p)]
  if not cands:
    top = f'{job_dir}/{jobname}_confidences.json'
    cands = [top] if os.path.exists(top) else []
  if cands:
    pae = json.load(open(cands[0])).get('pae')
    if pae is not None:
      return np.asarray(pae, dtype=float)
  return None

# BOTH MODES WRITE THE SAME FOLDER NOW, so an empty one means the run cell
# has not finished successfully -- there is no third possibility to tell
# apart, which is what this said before live_view learned to write outputs.
def _no_results_message(where):
  return (f'No results in {where}. Run the cell above first -- and if it'
          ' reported an error, that is the one to read.')

cifs = collect_models()
if not cifs:
  raise FileNotFoundError(_no_results_message(f'{job_dir}/'))
print(f'Loaded {len(cifs)} model(s) from {job_dir}/'
      + ('  (frames - press play)' if load_as_frames else '  (use the dropdown to switch)'))

viewer = py2Dmol.view(size=(viewer_size, viewer_size),
                      pae=True, autoplay=load_as_frames,
                      style="cartoon")
for i, cif in enumerate(cifs, start=1):
  pae = load_pae(cif)
  if load_as_frames:
    # use_biounit=False: py2Dmol 2.0 builds biological assemblies by default,
    # and what is on screen here should be what the model PREDICTED. A no-op
    # for our own output (no _pdbx_struct_assembly in it, checked) but not for
    # a file that carries one.
    viewer.add_pdb(cif, name='models', paes=pae, use_biounit=False)  # same name -> frames
  else:
    viewer.add_pdb(cif, name=f'rank_{i}', paes=pae, use_biounit=False)  # distinct names -> dropdown
viewer.show()


In [ ]:
#@title Quality metrics and plots
import json, os
import numpy as np
import matplotlib.pyplot as plt

conf_path = f'{OUTPUT_DIR}/{jobname}/{jobname}_confidences.json'
summ_path = f'{OUTPUT_DIR}/{jobname}/{jobname}_summary_confidences.json'

# BOTH MODES WRITE THE SAME FOLDER NOW, so an empty one means the run cell
# has not finished successfully -- there is no third possibility to tell
# apart, which is what this said before live_view learned to write outputs.
def _no_results_message(where):
  return (f'No results in {where}. Run the cell above first -- and if it'
          ' reported an error, that is the one to read.')

if not (os.path.exists(conf_path) and os.path.exists(summ_path)):
  raise FileNotFoundError(_no_results_message(f'{OUTPUT_DIR}/{jobname}/'))

with open(conf_path) as f:
  conf = json.load(f)
with open(summ_path) as f:
  summ = json.load(f)

plddts = np.array(conf.get('atom_plddts', conf.get('token_plddts', [])), dtype=float)
plddt_chain_ids = conf.get('atom_chain_ids', conf.get('token_chain_ids', []))   # pLDDT is per-ATOM
# ...and the fallback is a TOKEN list, which is shorter. Colouring per chain
# compares it elementwise against the per-atom pLDDTs, so a mismatch is a
# numpy broadcast error rather than a wrong picture; one line is better than
# a traceback in a plotting cell.
if len(plddt_chain_ids) != len(plddts):
  plddt_chain_ids = []
token_chain_ids = conf.get('token_chain_ids', [])                                # PAE is per-TOKEN
pae = np.array(conf.get('pae', []), dtype=float)

# ── Summary (ipTM is None for single-chain jobs — guard before formatting) ─
def fmt(v):
  return f'{v:.3f}' if isinstance(v, (int, float)) else 'n/a'

mean_plddt = summ.get('mean_plddt')
if mean_plddt is None and plddts.size:
  mean_plddt = float(np.mean(plddts))
iptm = summ.get('iptm')

print('=' * 38)
print(f'Mean pLDDT     : {fmt(mean_plddt)}')
print(f'pTM            : {fmt(summ.get("ptm"))}')
print(f'ipTM           : {fmt(iptm)}' + ('   (single chain — no interface)' if iptm is None else ''))
print(f'Ranking score  : {fmt(summ.get("ranking_score"))}')
print('=' * 38)

# ── Plots ───────────────────────────────────────────────────
has_pae = pae.ndim == 2 and pae.size > 0
ncols = 2 if has_pae else 1
fig, axes = plt.subplots(1, ncols, figsize=(13 if has_pae else 6.5, 4))
axes = np.atleast_1d(axes)

# pLDDT per residue — a line (coloured per chain when there is more than one)
ax = axes[0]
x = np.arange(len(plddts))
xmax = max(len(plddts) - 1, 1)
ax.set_xlim(0, xmax)
ax.set_ylim(0, 100)

unique_chains = list(dict.fromkeys(plddt_chain_ids))
if len(unique_chains) > 1:
  colors = plt.cm.tab10(np.linspace(0, 1, len(unique_chains)))
  tcid = np.array(plddt_chain_ids)
  for ch, col in zip(unique_chains, colors):
    y = np.where(tcid == ch, plddts, np.nan)   # NaN gaps keep chains as separate lines
    ax.plot(x, y, lw=1.5, color=col, label=f'Chain {ch}')
  for b in [i for i in range(1, len(plddt_chain_ids)) if plddt_chain_ids[i] != plddt_chain_ids[i-1]]:
    ax.axvline(b - 0.5, color='grey', lw=0.6, alpha=0.5)
  ax.legend(loc='lower right', fontsize=8)
else:
  ax.plot(x, plddts, lw=1.5, color='#1f77b4')

for y in (50, 70, 90):
  ax.axhline(y, ls='--', lw=0.7, color='grey', alpha=0.5)
  ax.text(xmax, y, f' {y}', va='center', ha='left', fontsize=7, color='grey')
ax.set_xlabel('Atom')
ax.set_ylabel('pLDDT')
ax.set_title('Predicted pLDDT per atom')

# PAE matrix
if has_pae:
  ax = axes[1]
  im = ax.imshow(pae, cmap='bwr', vmin=0, vmax=30, interpolation='nearest')
  plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='PAE (Å)')
  if token_chain_ids:
    for b in [i for i in range(1, len(token_chain_ids)) if token_chain_ids[i] != token_chain_ids[i-1]]:
      ax.axhline(b - 0.5, c='black', lw=0.8)
      ax.axvline(b - 0.5, c='black', lw=0.8)
  ax.set_xlabel('Scored residue')
  ax.set_ylabel('Aligned residue')
  ax.set_title('Predicted Aligned Error (PAE)')

plt.tight_layout()
plt.show()


In [ ]:
#@title Download results
from google.colab import files
import os

# BOTH MODES WRITE THE SAME FOLDER NOW, so an empty one means the run cell
# has not finished successfully -- there is no third possibility to tell
# apart, which is what this said before live_view learned to write outputs.
def _no_results_message(where):
  return (f'No results in {where}. Run the cell above first -- and if it'
          ' reported an error, that is the one to read.')

if not os.path.isdir(f'{OUTPUT_DIR}/{jobname}'):
  raise FileNotFoundError(_no_results_message(f'{OUTPUT_DIR}/{jobname}/'))

results_zip = f'{jobname}.result.zip'
# Checked: a failed zip otherwise reaches files.download as 'cannot find
# file', which reads like a Colab fault rather than an empty results folder.
if os.system(f'zip -r -q {results_zip} {OUTPUT_DIR}/{jobname}') != 0 \
    or not os.path.exists(results_zip):
  raise RuntimeError(f'could not archive {OUTPUT_DIR}/{jobname} -- has the'
                     ' run cell finished?')
print(f'{results_zip}  ({os.path.getsize(results_zip) / 1e6:.1f} MB)')
files.download(results_zip)


# Instructions <a name="Instructions"></a>

**Quick start:** pick a **model** in the install cell, fill in the sequence(s), then
**Runtime → Run all**. The first run downloads that model's weights; later runs reuse
them, and each model has its own cache so switching back is instant.

## Options

The run cell's fields. A field for the other family is ignored, not an error,
and **0 means the model's own default**.

| field | family | what it does |
|---|---|---|
| `num_recycles` | either | Refinement passes. 3 here, which is AlphaFold 2's own number and ColabFold's habit; AlphaFold 3's own is 10, so this trades a little accuracy on hard targets for seven fewer trunk passes. Raise it for anything unconverged. |
| `num_msa` | either | MSA rows the trunk reads, 512 here. AlphaFold 2's second, "extra" stack follows at twice this. Barely a speed dial — 1 against 1024 measured 0.4% of runtime at 512 tokens — but it is the knob for an out-of-memory. |
| `live_view` | either | Fold in the notebook's own process and stream it: AlphaFold 3 draws every denoising step and a contact map per recycle, AlphaFold 2 draws every recycle, being the only intermediate structure it has. A frame is drawn one step behind the maths, so the fold never waits for the picture. It writes the same output folder as a normal run. **Not bit-identical**: each stage restarts haiku's rng, so a structure moves about as much as a different seed would (1.52 Å, against 0.47–1.35 Å between seeds). Untick it for a bit-identical run. |
| `num_diffusion_samples` | AF3 | Structures per seed, so the total is seeds × samples. Both modes fold all of them; the live view animates the first. |
| `diffusion_steps` | AF3 | Denoising steps, 32 here against AlphaFold 3's own 200. Below about 20 the sampler does not land — ten gives 5.91 Å on 6MRR against a Cα–Cα of 8.40. |
| `af2_num_models` | AF2 | AlphaFold 2 ships five separately trained parameter sets — five models, not five seeds — and they appear as the samples of a seed. Running several and letting the ranking sort them is what ColabFold's `num_models` does. |

## Models

Ported weights come from [sokrypton/af3-any-model](https://huggingface.co/sokrypton/af3-any-model).
"Tower" is a protein language model fetched separately on first use.

| model | weights | licence | notes |
|---|---|---|---|
| `openbind0` | [OpenFold3 v0.5.0 "OpenBind"](https://github.com/aqlaboratory/openfold-3/releases/tag/v0.5.0) (AlQuraishi Lab) | Apache-2.0 | The current release, and the default here. |
| `openfold3` | [OpenFold3 preview-2](https://github.com/aqlaboratory/openfold) (AlQuraishi Lab) | Apache-2.0 | The earlier preview, kept because earlier results used it. |
| `boltz2` | [Boltz-2](https://github.com/jwohlwend/boltz) (Wohlwend et al.) | MIT | Strong on ligands; keeps a modified residue as one token. |
| `protenix2` | [Protenix-v2](https://github.com/bytedance/Protenix) (ByteDance) | Apache-2.0 | The widest trunk here (pair 256), so the slowest. |
| `rosettafold3` | [RoseTTAFold3](https://github.com/RosettaCommons/foundry) (RosettaCommons) | BSD-3-Clause | Carries chirality features; handles D-amino acids. |
| `chai1` | [chai-1](https://github.com/chaidiscovery/chai-lab) (Chai Discovery) | Apache-2.0 | Folds from ESM2 3B, fetched and run automatically. |
| `intellifold2` | [IntelliFold-v2](https://huggingface.co/intelligenAI/intellifold) (IntelliGen-AI) | Apache-2.0 | Widened channels (pair 512), largest ported download. |
| `opendde` | [OpenDDE](https://huggingface.co/aurekaresearch/OpenDDE) (Aureka Research) | Apache-2.0 | Runs its diffusion on an expanded structural-token set. |
| `esmfold2` | [ESMFold2](https://huggingface.co/biohub/ESMFold2) (Chan Zuckerberg Biohub) | MIT | Folds from ESM-C instead of an MSA — single sequence, no search. |
| `esmfold2_lm600m` | ESMFold2, 600M tower | MIT | No confidence head. |
| `esmfold2_lm300m` | ESMFold2, 300M tower | MIT | No confidence head. |
| `af2_ptm` | AlphaFold 2 monomer pTM (DeepMind) | CC BY 4.0 | **Protein only** — a ligand or nucleotide raises rather than quietly folding the rest. Templates use the model_1/model_2 parameter sets. |
| `af2_multimer` | AlphaFold 2 multimer v3 (DeepMind) | CC BY 4.0 | Protein only, as above. |
| `alphafold3` | Google DeepMind's own parameters | [AF3 terms of use](https://github.com/google-deepmind/alphafold3/blob/main/WEIGHTS_TERMS_OF_USE.md) | DeepMind's public release, downloaded on first use (~1 GB). Its terms govern the weights and the outputs; run_alphafold prints them at startup. |
---

## Input

Each molecule type has its own box; within a box, separate chains with `:`.

| box | contents | example |
|---|---|---|
| **protein** | amino-acid sequence(s) | `MKTAY...` or `SEQ1:SEQ2` |
| **dna** | DNA sequence(s) | `CGCGAATTCGCG` |
| **rna** | RNA sequence(s) | `GCGGAUUUA` |
| **ligand_ccd** | ligand(s) by PDB CCD code | `ATP:MG:HEM` |
| **ligand_smiles** | ligand(s) by SMILES | `CC(=O)Oc1ccccc1C(=O)O` |

Mix boxes freely to build a complex. Chain IDs A, B, C… follow AlphaFold 3's canonical
order (protein → RNA → DNA → ligand). Identical protein sequences are merged, so
`SEQ:SEQ` is a homodimer. Sequences and CCD codes are upper-cased; **SMILES are left
as typed**. Whitespace and extra colons are forgiven (`SEQ1::::SEQ2` = `SEQ1:SEQ2`) —
which is also why an atom-mapped SMILES containing `:` needs a raw AF3 JSON instead.

**seeds**: comma-separated, one prediction each (`1,2,3`). Junk and duplicates are
dropped. **msa_mode**: `mmseqs2_server` queries the public
[ColabFold](https://colabfold.mmseqs.com/) API (protein only — RNA/DNA always run
MSA-free); `single_sequence` skips it, faster and less accurate.

## Output

| file | contents |
|---|---|
| `*.cif` | Best-ranked structure. B-factor = pLDDT (0–100). |
| `*_confidences.json` | Per-residue pLDDT, PAE matrix, contact probabilities. |
| `*_summary_confidences.json` | Mean pLDDT, pTM, ipTM, ranking score. |
| `*_ranking_scores.csv` | Every seed × sample combination. |
| `seed-N_sample-M/` | One directory per prediction. |
| `TERMS_OF_USE.md` | The licence for whichever weights you ran. |

pLDDT above 90 is very high, 70–90 reliable backbone, 50–70 doubtful, below 50 likely
disordered or wrong. Lower PAE means two residues are confidently placed *relative to
each other*, which is what to read for an interface. ipTM above 0.8 is a well-defined
complex interface, and is `n/a` for a single chain — there is no interface to score.

## Troubleshooting

**OOM**: shorter sequence, or a larger GPU (`Runtime → Change runtime type`).
**MSA server timeout**: the public server is rate-limited — retry, or use
`single_sequence`. **Download popup blocked**: disable your ad blocker.

## Licence

The AlphaFold 3 **source code** is [Apache 2.0](https://www.apache.org/licenses/LICENSE-2.0);
the **weights** are each their own, as listed in the model table. Outputs from the seven
Apache/MIT/BSD-licensed ported models are **not** subject to DeepMind's AlphaFold 3 Output
Terms of Use and may be used freely, including commercially. `alphafold3` is the exception:
its parameters and outputs carry DeepMind's own terms. Every run writes a
`TERMS_OF_USE.md` naming the licence that actually applies to it.

## Bugs / feedback

https://github.com/sokrypton/alphafold3/issues
